In [63]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [64]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


### Q1. Missing values

In [65]:
df.columns[df.isnull().sum().values > 0]

Index(['horsepower', 'acceleration'], dtype='str')

### Q2. Median for horsepower

In [66]:
print(df.horsepower.median())

254.0


### Q3. Filling NAs

In [102]:
n = len(df)

n_val = int(len(df) * 0.2)
n_test = int(len(df) * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)

np.random.seed(42)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val   = df.iloc[idx[n_train : n_train + n_val]].reset_index(drop=True)
df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [105]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'num_cylinders', 'acceleration']
categorical_variables = ['origin', 'fuel_type', 'drivetrain', 'num_doors']

categories = {}

for c in categorical_variables:
    categories[c] = list(df[c].value_counts().head().index)

def rmse(y, y_pred) :
    se = (y - y_pred) ** 2
    mse = se.mean()

    return np.sqrt(mse)

def train_linear_regression_reg(X, y, r=0.001):
    ones = np. ones (X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def prepare_X(df, zero=True) :
    df = df.copy()
    df['age'] = 2024 - df.model_year
    features = base + ['age']

    for c, values in categories.items():
        for v in values:
            df['%s_%s' % (c, v)] = (df[c] == v).astype(int)
            features.append('%s_%s' % (c, v))

    df_num = df[features]

    hp_mean = df_train['horsepower'].mean()

    if zero:
        df_num = df_num.fillna(0)
    else:
        df_num = df_num.fillna(hp_mean)

    X = df_num.values
    
    return X

# With Zero
X_train = prepare_X(df_train, True)
w0, w = train_linear_regression_reg(X_train, y_train, 0.01)

X_val = prepare_X(df_val, zero=True)
y_pred = w0 + X_val.dot(w)

with_zero = rmse(y_val, y_pred)

print("With zero: ", round(with_zero, 3))

# With mean
X_train = prepare_X(df_train, False)
w0, w = train_linear_regression_reg(X_train, y_train, 0.01)

X_val = prepare_X(df_val, zero=True)
y_pred = w0 + X_val.dot(w)

with_mean = rmse(y_val, y_pred)

print("With mean: ", round(with_mean, 3))

print(with_mean > with_zero)


With zero:  0.054
With mean:  0.056
True


### Q4. Best regularization

In [106]:
mini = 10e9
mini_r = 0

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r)

    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)

    if(score < mini):
        mini = score
        mini_r = r

    print(r, w0, score)

print("Best: ", mini, " r: ", mini_r)

0 -1341534895329951.5 18.415246326494866
0.01 1.7883565582597112 0.053642776001361095
0.1 1.7836971576759417 0.05364453702653662
1 1.7384248532502227 0.053950114280868086
5 1.5624123606291147 0.05977685786322151
10 1.3872670909365739 0.0712537015902205
100 0.4670489779923842 0.16380527487703872
Best:  0.053642776001361095  r:  0.01


### Q5. RMSE standard deviation

In [107]:
scores = []

for s in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    idx = np.arange(n)
    np.random.seed(s)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val   = df.iloc[idx[n_train : n_train + n_val]].reset_index(drop=True)
    df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

    y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
    y_val   = np.log1p(df_val.fuel_efficiency_mpg.values)

    X_train = df_train[base].fillna(0).values
    X_val   = df_val[base].fillna(0).values

    w0, w = train_linear_regression_reg(X_train, y_train, r=0)

    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val, y_pred)
    scores.append(score)

std_val = round(np.std(scores), 3)
print("RMSE Scores:", scores)
print("Standard Deviation:", std_val)

RMSE Scores: [np.float64(0.08562411964450851), np.float64(0.0833213256656831), np.float64(0.08192548386145608), np.float64(0.08436518359143826), np.float64(0.08227422746270434), np.float64(0.08566865336265575), np.float64(0.08575157950112604), np.float64(0.0834822381568582), np.float64(0.08398227823379578), np.float64(0.08350663187230237)]
Standard Deviation: 0.001


### Q6.  Evaluation on test

In [96]:
df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)

X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])

w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)

rmse(y_test, y_pred)

car = df_test.iloc[20].to_dict()

df_small = pd.DataFrame([car])
X_small = prepare_X(df_small)
y_pred = w0 + X_small.dot(w)
y_pred = y_pred[0]
print(y_pred)
print(y_test[20])
print(np.expm1(y_pred))
print(np.expm1(y_test[20]))

3.3829423593313463
3.380994674344636
28.45731773893963
28.400000000000002
